In [1]:
import h5py
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import norm
from tqdm.auto import tqdm
from scipy.optimize import OptimizeWarning
import warnings

/home/lenka-hake/Documents/ALS/ALS-Assignment-3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Constants

In [2]:
number_of_anchors = 137
prediction_horizon = 16
number_of_bins = 1000
anchor_index_array = np.arange(number_of_anchors)
training_size_array = np.ceil(16 * (2 ** (anchor_index_array / 8.0))).astype(float)
bin_edge_array = np.linspace(0.0, 1.0, number_of_bins + 1)

## Data Loading

In [3]:
def load_lcdb_file(file_path):
    with h5py.File(file_path, "r") as file_handle:
        error_rate_array = np.asarray(file_handle["error rate"][:], dtype=float)
        learner_array = np.asarray(file_handle["learner"][:]).reshape(-1)

    if learner_array.dtype.kind in {"S", "U", "O"}:

        cleaned_learner_array = []
        for learner_value in learner_array:
            if isinstance(learner_value, bytes):
                cleaned_learner_array.append(learner_value.decode())
            else:
                cleaned_learner_array.append(str(learner_value))
        learner_array = np.asarray(cleaned_learner_array)

    else:

        learner_array = learner_array.astype(int)

    return error_rate_array, learner_array

## Helpers

In [4]:
def get_contiguous_prefix(curve_array):
    anchor_list = []
    value_list = []

    for anchor_index, error_value in enumerate(curve_array):
        if np.isnan(error_value):
            break

        anchor_list.append(anchor_index)
        value_list.append(float(error_value))

    if len(anchor_list) == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    anchor_array = np.asarray(anchor_list, dtype=int)
    value_array = np.asarray(value_list, dtype=float)
    return anchor_array, value_array

## Parametric Models

In [5]:
def power_law_model(training_size, a, b, c):
    safe_training_size = np.maximum(training_size, 1.0)
    safe_exponent = np.clip(-b, -20.0, 20.0)
    return a * np.exp(safe_exponent * np.log(safe_training_size)) + c

def logarithmic_model(training_size, a, b):
    safe_training_size = np.maximum(training_size, 1.0)
    return a + b * np.log(safe_training_size)

def inverse_model(training_size, a, b, c):
    safe_denominator = np.maximum(training_size + b, 1e-6)
    return c + a / safe_denominator

In [6]:
models = {
    "power_law": {
        "function": power_law_model,
        "initial_parameter_function": lambda x_values, y_values: [ max(y_values[0] - y_values[-1], 0.001), 0.3, float(np.clip(y_values[-1], 0.0, 1.0))],
        "bounds": ([0.0, -5.0, 0.0], [2.0, 10.0, 1.0])
    },
    "logarithmic": {
        "function": logarithmic_model,
        "initial_parameter_function": lambda x_values, y_values: [float(np.clip(y_values[0], 0.0, 1.0)), -0.05],
        "bounds": ([-1.0, -2.0], [2.0, 2.0])
    },
    "inverse": {
        "function": inverse_model,
        "initial_parameter_function": lambda x_values, y_values: [max(y_values[0] - y_values[-1], 0.001), 1.0, float(np.clip(y_values[-1], 0.0, 1.0))],
        "bounds": ([0.0, -1000.0, 0.0], [5.0, 10000.0, 1.0])
    }
}

## Fitting and Prediction

In [7]:
def fit_and_predict(anchor_indexes, errors, target_anchor_index, model_name):

    model_specification = models[model_name]
    model_function = model_specification["function"]
    parameter_bounds = model_specification["bounds"]
    minimum_points = 4

    if len(errors) == 0:
        return 0.5

    if len(errors) < minimum_points:
        return float(np.clip(errors[-1], 0.0, 1.0))

    x_values = training_size_array[anchor_indexes]
    y_values = errors.astype(float)
    target_value = training_size_array[target_anchor_index]

    mask = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = x_values[mask]
    y_values = y_values[mask]

    if len(y_values) < minimum_points:
        return float(np.clip(errors[-1], 0.0, 1.0))

    try:
        initial_parameters = model_specification["initial_parameter_function"](x_values, y_values)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", OptimizeWarning)
            warnings.simplefilter("ignore", RuntimeWarning)

            fit_result = curve_fit(
                model_function,
                x_values,
                y_values,
                p0=initial_parameters,
                bounds=parameter_bounds,
                maxfev=5000
            )

        fitted_parameters = fit_result[0]
        predicted_value = model_function(np.array([target_value], dtype=float), *fitted_parameters)[0]

        if not np.isfinite(predicted_value):
            predicted_value = errors[-1]

        predicted_value = float(np.clip(predicted_value, 0.0, 1.0))
        return predicted_value

    except Exception:
        return float(np.clip(errors[-1], 0.0, 1.0))

## Validation Example Generation

In [8]:
def build_examples(
    curve_array,
    learner_array,
    minimum_prefix_points=6,
    maximum_examples_per_curve=1,
    random_seed=42
):

    random_generator = np.random.default_rng(random_seed)
    examples = []

    for curve_index in range(len(curve_array)):

        curve = curve_array[curve_index]
        learner = learner_array[curve_index]
        anchors_index_array, error_array = get_contiguous_prefix(curve)
        prefix_length = len(anchors_index_array)

        if prefix_length < minimum_prefix_points + prediction_horizon:
            continue

        possible_prefix_endpoints = np.arange(minimum_prefix_points - 1, prefix_length - prediction_horizon)

        if len(possible_prefix_endpoints) > maximum_examples_per_curve:
            selected_endpoints = random_generator.choice(
                possible_prefix_endpoints,
                size=maximum_examples_per_curve,
                replace=False
            )
        else:
            selected_endpoints = possible_prefix_endpoints

        for prefix_endpoint in selected_endpoints:

            anchor_indexes = anchors_index_array[:prefix_endpoint + 1]
            errors = error_array[:prefix_endpoint + 1]

            target_anchor_index = anchors_index_array[prefix_endpoint + prediction_horizon]
            target_error_value = error_array[prefix_endpoint + prediction_horizon]

            examples.append({
                "learner": learner,
                "observed_anchor_index_array": anchor_indexes,
                "observed_error_array": errors,
                "target_anchor_index": target_anchor_index,
                "target_error_value": target_error_value
            })

    return examples

## Model Evaluation

In [9]:
def evaluate_models(examples, model_names):

    result = []
    for example in tqdm(examples):

        learner = example["learner"]
        target_error_value = example["target_error_value"]

        for model_name in model_names:

            prediction = fit_and_predict(
                example["observed_anchor_index_array"],
                example["observed_error_array"],
                example["target_anchor_index"],
                model_name
            )

            absolute_error = abs(prediction - target_error_value)

            result.append({
                "learner": learner,
                "model_name": model_name,
                "absolute_error": absolute_error,
                "success": 1
            })

    return pd.DataFrame(result)

## Determining the Best Model for Each Learner

In [10]:
def select_best_model_per_learner(results_dataframe):

    grouped_dataframe = (
        results_dataframe
        .groupby(["learner", "model_name"])
        .agg(mean_absolute_error=("absolute_error", "mean"))
        .reset_index()
    )

    best_models = (
        grouped_dataframe
        .sort_values(["learner", "mean_absolute_error"])
        .groupby("learner")
        .first()
        .reset_index()
    )

    return dict(zip(best_models["learner"], best_models["model_name"])), best_models

## Evaluation of Model Residuals

In [11]:
def compute_residual_standard_deviation(examples, best_models):

    residual_values = {}
    for example in tqdm(examples):

        learner = example["learner"]
        model_name = best_models[learner]

        prediction = fit_and_predict(
            example["observed_anchor_index_array"],
            example["observed_error_array"],
            example["target_anchor_index"],
            model_name
        )

        residual = prediction - example["target_error_value"]
        key = (learner, model_name)

        if key not in residual_values:
            residual_values[key] = []

        residual_values[key].append(residual)

    residual_deviations = {}

    for key, residual_list in residual_values.items():
        residual_array = np.array(residual_list, dtype=float)
        standard_deviation = float(np.std(residual_array))

        if not np.isfinite(standard_deviation) or standard_deviation < 0.01:
            standard_deviation = 0.01

        residual_deviations[key] = standard_deviation

    return residual_deviations

## Conversion of Gaussian Distribution to Bin Probabilities

In [12]:
def convert_to_bin_probabilities(mean_value, standard_deviation):

    if standard_deviation is None or not np.isfinite(standard_deviation) or standard_deviation < 0.01:
        standard_deviation = 0.03

    bin_edges = np.linspace(0.0, 1.0, 1001)
    probability_array = (norm.cdf(bin_edges[1:], loc=mean_value, scale=standard_deviation)
                         - norm.cdf(bin_edges[:-1], loc=mean_value, scale=standard_deviation))
    probability_array = np.clip(probability_array, 0.0, None)
    probability_sum = probability_array.sum()

    if probability_sum <= 0.0 or not np.isfinite(probability_sum):
        probability_array = np.ones(1000, dtype=float) / 1000.0
    else:
        probability_array = probability_array / probability_sum

    return probability_array

## Test Set Probabilities

In [13]:
def predict_test_probabilities(
    test_curves,
    test_learners,
    best_models,
    residual_deviations
):
    probability_row_list = []
    for curve_index in tqdm(range(len(test_curves))):

        curve = test_curves[curve_index]
        learner = test_learners[curve_index]
        anchor_indexes, errors = get_contiguous_prefix(curve)

        if len(anchor_indexes) == 0:
            probability_row_list.append(np.ones(1000, dtype=float) / 1000.0)
            continue

        target_anchor_index = anchor_indexes[-1] + 16

        if target_anchor_index >= len(training_size_array):
            target_anchor_index = len(training_size_array) - 1

        model_name = best_models.get(learner, "power_law")
        prediction = fit_and_predict(anchor_indexes, errors, target_anchor_index, model_name)
        standard_deviation = residual_deviations.get((learner, model_name), 0.03)
        probability_array = convert_to_bin_probabilities(prediction,standard_deviation)
        probability_row_list.append(probability_array)

    probabilities = np.vstack(probability_row_list)

    return probabilities

## Submission Generation

In [17]:
def write_submission_file(probabilities, output_path):
    submission_dataframe = pd.DataFrame(probabilities, columns=[f"bin_{bin_index}" for bin_index in range(1000)])
    submission_dataframe.insert(0, "id", np.arange(len(submission_dataframe)))
    submission_dataframe.to_csv(output_path, index=False)
    print("Saved submission to:", output_path)

## Subsampling Helper

In [15]:
def sample_curve_subset_by_learner(
    curve_array,
    learner_array,
    maximum_curves_per_learner,
    random_seed=42
):
    random_generator = np.random.default_rng(random_seed)
    selected_index_list = []
    unique_learner_array = np.unique(learner_array)

    for learner in unique_learner_array:
        learner_index_array = np.where(learner_array == learner)[0]

        if len(learner_index_array) <= maximum_curves_per_learner:
            chosen_index_array = learner_index_array
        else:
            chosen_index_array = random_generator.choice(
                learner_index_array,
                size=maximum_curves_per_learner,
                replace=False
            )

        selected_index_list.extend(chosen_index_array.tolist())

    selected_index_array = np.array(sorted(selected_index_list), dtype=int)
    subset_curve_array = curve_array[selected_index_array]
    subset_learner_array = learner_array[selected_index_array]
    return subset_curve_array, subset_learner_array, selected_index_array

## Full Pipeline

In [16]:
train_curve_array, train_learner_array = load_lcdb_file("data/LCDB11_ER_train.hdf5")
test_curve_array, test_learner_array = load_lcdb_file("data/LCDB11_ER_eval.hdf5")

subset_train_curve_array, subset_train_learner_array, subset_index_array = sample_curve_subset_by_learner(
    train_curve_array,
    train_learner_array,
    maximum_curves_per_learner=100,
    random_seed=42
)

example_list = build_examples(
    train_curve_array, #subset_train_curve_array
    train_learner_array, #subset_train_learner_array
    minimum_prefix_points=6,
    maximum_examples_per_curve=3,
    random_seed=42
)


print("Number of validation examples:", len(example_list))

model_name_list = list(models.keys())
result_dataframe = evaluate_models(example_list, model_name_list)

best_model_dictionary, best_model_rows = select_best_model_per_learner(result_dataframe)

print("Best models per learner:")
print(best_model_rows)

residual_dictionary = compute_residual_standard_deviation(example_list, best_model_dictionary)

probability_matrix = predict_test_probabilities(
    test_curve_array,
    test_learner_array,
    best_model_dictionary,
    residual_dictionary
)

print("Probability matrix shape:", probability_matrix.shape)
print("First row sum:", probability_matrix[0].sum())

Number of validation examples: 120384


100%|██████████| 120384/120384 [21:22<00:00, 93.90it/s] 


Best models per learner:
    learner model_name  mean_absolute_error
0         0  power_law             0.054462
1         1  power_law             0.070211
2         2  power_law             0.058101
3         3  power_law             0.045881
4         4  power_law             0.056822
5         5  power_law             0.074261
6         6  power_law             0.045547
7         7  power_law             0.062951
8         8  power_law             0.055531
9         9  power_law             0.049378
10       10  power_law             0.053217
11       11  power_law             0.066171


100%|██████████| 1000/1000 [00:04<00:00, 213.91it/s]


Probability matrix shape: (1000, 1000)
First row sum: 1.0
Saved submission to: submission.csv


In [18]:
write_submission_file(probability_matrix, "submission.csv")

Saved submission to: submission.csv
